# 🎬 VideoFX — gerador rápido de vídeos (estilo MoneyPrinter)

Roda 100% no navegador: narração TTS neural em português, imagens de fundo, legendas sincronizadas, música e efeito Ken Burns.

**Colab:** `Ambiente de execução → Executar tudo` *(ou clique nos botões ▶ de cada célula)*  
**Kaggle:** *File → Import Notebook → from GitHub* + Settings → **Internet: ON**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jushirambo/videofx/blob/main/videofx_colab.ipynb)


## 1️⃣ Instalação (~1 min)


In [ ]:
%%capture
!git clone https://github.com/Jushirambo/videofx.git /content/videofx
%cd /content/videofx
!pip install -q -r requirements.txt
import os, sys
sys.path.insert(0, '/content/videofx')
print('✅ pronto'))


## 2️⃣ Configure o vídeo
Edite os campos abaixo e rode a célula (▶).

**Roteiro:** deixe `ROTULO` vazio para gerar automático (template local) — ou escolha um provedor de IA:
- **openrouter**: use modelos como `meta-llama/llama-3.3-70b-instruct` (chave em [openrouter.ai/keys](https://openrouter.ai/keys))
- **huggingface**: modelos como `meta-llama/Llama-3.1-8B-Instruct` (token em [hf.co/settings/tokens](https://huggingface.co/settings/tokens))

Também funciona sem preencher `api_key` se você definir a variável de ambiente `OPENROUTER_API_KEY` ou `HF_TOKEN`.

Ou cole suas falas em `ROTULO`, uma por cena.


In [ ]:
#@title ⚙️ Configuração do vídeo { display-mode: "form" }
topic = "Fatos incríveis sobre o universo" #@param {type:"string"}
title = "" #@param {type:"string"}
n_scenes = 5 #@param {type:"slider", min:2, max:8, step:1}
aspect = "9:16" #@param ["9:16", "16:9", "1:1"]
provider = "template" #@param ["template", "openrouter", "huggingface"]
model = "" #@param {type:"string"}
api_key = "" #@param {type:"string"}
script_style = "viral" #@param ["viral", "educativo", "storytelling"]
transitions = "fade,slideleft,zoomin,smoothleft,circleopen" #@param {type:"string"}
transition_duration = 0.5 #@param {type:"slider", min:0.1, max:1.2, step:0.1}
subtitle_style = "pop" #@param ["pop", "box", "clean"]
music_mood = "calm" #@param ["calm", "energetic"]
particles = True #@param {type:"boolean"}
shine = True #@param {type:"boolean"}
vignette = True #@param {type:"boolean"}
color_boost = True #@param {type:"boolean"}
progress_bar = True #@param {type:"boolean"}
ROTULO = "" #@param {type:"string"}

import json, os
cfg = {
    "topic": topic,
    "title": title or None,
    "aspect": aspect,
    "provider": provider,
    "model": model or None,
    "api_key": api_key or os.environ.get("OPENROUTER_API_KEY") or os.environ.get("HF_TOKEN"),
    "script_style": script_style,
    "transitions": transitions,
    "transition_duration": transition_duration,
    "subtitle_style": subtitle_style,
    "music_mood": music_mood,
    "particles": particles,
    "shine": shine,
    "vignette": vignette,
    "color_boost": color_boost,
    "progress_bar": progress_bar,
    "allow_ai_images": True,
    "output_name": "video_final.mp4",
}
if not title.strip():
    cfg.pop("title", None)
if provider == "template":
    cfg["n_scenes"] = n_scenes
if ROTULO.strip():
    cfg["scenes"] = [l.strip() for l in ROTULO.splitlines() if l.strip()]
    cfg.pop("n_scenes", None)
json.dump(cfg, open("config.json", "w"), ensure_ascii=False, indent=2)
print("config salva ✅",
      "(roteiro manual)" if ROTULO.strip() else
      ("(roteiro IA: " + provider + ")" if provider != "template" else "(roteiro automático)"))


## 3️⃣ Gere o vídeo (~2–4 min)
 imagem + narracão + música + legendas são geradas nesta célula.


In [ ]:
!python main.py --config config.json


## 4️⃣ Assista e baixe
O vídeo aparece abaixo e um download é disparado no navegador (no Kaggle, use o painel *Output*).


In [ ]:
from IPython.display import Video, display
display(Video("out/video_final.mp4", embed=True, height=560, width=320))
try:
    from google.colab import files
    files.download("out/video_final.mp4")
except ImportError:
    print("Kaggle: baixe em Data ▸ Output ▸ video_final.mp4"))


## 🎛️ (Opcional) Interface interativa no navegador
Rode para abrir um painel Gradio com link público (`*.gradio.live`) durante a sessão.


In [ ]:
%%capture
!pip install -q gradio


In [ ]:
import json, subprocess, gradio as gr, os

def gerar(topico, titulo, aspect, provider, model, api_key, style, transitions, td,
          sub_style, mood, chars, subsize, particulas, brilho, vineta, cores, barra, rotulo):
    cfg = {"topic": topico, "aspect": aspect, "provider": provider,
           "model": model or None, "api_key": api_key or None, "script_style": style,
           "transitions": transitions, "transition_duration": td,
           "subtitle_style": sub_style, "music_mood": mood,
           "subtitle_max_chars": int(chars), "subtitle_size": int(subsize),
           "particles": particulas, "shine": brilho, "vignette": vineta,
           "color_boost": cores, "progress_bar": barra,
           "output_name": "video_final.mp4"}
    if titulo.strip():
        cfg["title"] = titulo
    if rotulo.strip():
        cfg["scenes"] = [l.strip() for l in rotulo.splitlines() if l.strip()]
    else:
        cfg["n_scenes"] = 5
    json.dump(cfg, open("config.json", "w"), ensure_ascii=False)
    p = subprocess.run(["python", "main.py", "--config", "config.json"], capture_output=True, text=True)
    if p.returncode != 0:
        raise gr.Error("falhou: " + p.stderr[-300:])
    return "out/video_final.mp4", p.stdout[-500:]

iface = gr.Interface(
    fn=gerar,
    inputs=[gr.Textbox(label="Tema", value="Fatos incríveis sobre o universo"),
            gr.Textbox(label="Título (opcional)"),
            gr.Radio(["9:16", "16:9", "1:1"], value="9:16", label="Formato"),
            gr.Radio(["template", "openrouter", "huggingface"], value="template", label="Roteiro"),
            gr.Textbox(label="Modelo (opcional)"),
            gr.Textbox(label="API key (opcional)", type="password"),
            gr.Radio(["viral", "educativo", "storytelling"], value="viral", label="Estilo do roteiro"),
            gr.Textbox(label="Transições (separadas por vírgula)", value="fade,slideleft,zoomin,smoothleft,circleopen"),
            gr.Slider(0.1, 1.2, value=0.5, step=0.1, label="Duração da transição (s)"),
            gr.Radio(["pop", "box", "clean"], value="pop", label="Estilo das legendas"),
            gr.Radio(["calm", "energetic"], value="calm", label="Música"),
            gr.Slider(18, 44, value=26, step=1, label="Máx. caracteres da legenda"),
            gr.Slider(40, 96, value=64, step=4, label="Tamanho da legenda"),
            gr.Checkbox(True, label="Partículas"),
            gr.Checkbox(True, label="Brilho (shine)"),
            gr.Checkbox(True, label="Vinheta"),
            gr.Checkbox(True, label="Realce de cores"),
            gr.Checkbox(True, label="Barra de progresso"),
            gr.Textbox(lines=4, label="ROTULO (uma fala por linha; vazio = automático)")],
    outputs=[gr.Video(label="Vídeo gerado"), gr.Textbox(label="Log")],
    title="🎬 VideoFX",
)
iface.launch(share=True, debug=True)


## ℹ️ Como funciona
Roteiro (template local ou IA via OpenRouter/HuggingFace) → TTS neural pt‑BR (Piper, voz baixada no 1º run) → imagem por cena (IA gratuita / fundo procedural) → ffmpeg: Ken Burns + transições xfade + legendas pop/box/clean + título animado + partículas com glow + varredura de brilho + vinheta + barra de progresso + música procedural.

Código: [github.com/Jushirambo/videofx](https://github.com/Jushirambo/videofx)
